In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from collections import Counter
print("finished")

finished


**Table of contents**<a id='toc0_'></a>

- [KNN 算法](#toc1_1_)
  - [逻辑回归模型](#toc1_2_)
  - [测试逻辑回归模型](#toc1_3_)

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->


## <a id='toc1_1_'></a>[KNN 算法](#toc0_)

原理：KNN 算法是一种基于距离度量的分类算法，其基本思想是：如果一个样本在特征空间中的 k 个最邻近的样本中的大多数属于某一类，则该样本也属于这个类。KNN 算法的实现过程可以分为以下几个步骤：

1. 计算样本之间的距离：首先需要计算样本之间的距离，常用的距离计算方法有欧式距离、曼哈顿距离、切比雪夫距离等。
2. 确定 k 值：KNN 算法的关键参数 k，即邻近的样本的数量。一般来说，k 值的选择对 KNN 算法的精度和效率都有很大的影响。
3. 确定类别：KNN 算法根据 k 个邻近样本的类别，决定待分类样本的类别。
4. 实现 KNN 算法：KNN 算法的实现过程可以分为以下几个步骤：
   - 1）计算待分类样本与样本库中每个样本之间的距离。
   - 2）按照距离递增次序排序。
   - 3）选取与待分类样本距离最小的 k 个样本。
   - 4）确定待分类样本的类别。


In [ ]:
class KNN:
    def __init__(self, k = 3):
        self.k = k # 取的个数
         # 训练集和标签
        self.X_train = None
        self.y_train = None
       

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def getDist(self, x1, x2):
        # return np.sqrt(np.sum(x1 - x2))
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    # 计算距离,给定单个数据点，返回类别
    def predict_one(self, x):
        distances = []
        for i in range(len(self.X_train)):
            dist = self.getDist(x, self.X_train[i])
            # 存储距离和标签
            distances.append((dist, self.y_train[i]))
            # 排序
        distances.sort(key=lambda x: x[0])
        # 取前k个
        k_neighbors = distances[:self.k]
        # 统计出现频率最高的标签
        # labels = max(k_neighbors, key=lambda x: x[1])[1]
        # 第二种实现方法
        labels = [label for _ , label in k_neighbors]
        labels = Counter(labels).most_common(1)[0][0]
        return labels
    
    def predict(self, X):
        y_pred = []
        for x in X:
            y_pred.append(self.predict_one(x))
        return y_pred
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return np.sum(y_pred == y) / len(y)

In [38]:
def test_KNN():
    knn = KNN()
    X_train = np.array([[1, 2], [2, 3], [3, 1], [4, 3], [5, 2],
                        [6, 4], [7, 5], [8, 6], [9, 7], [10, 8]])
    y_train = np.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1])
    X_test = np.array([[1, 1], [6, 5], [7, 6], [8, 7], [9, 8]])
    # knn.fit(X_train, y_train)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    print(y_pred)
    
test_KNN()

[0, 1, 1, 1, 1]


## <a id='toc1_2_'></a>[逻辑回归模型](#toc0_)

逻辑回归模型是一种分类模型，它可以用来预测某个变量的取值为 0 或 1，或者是某一事件发生的概率。

逻辑回归模型的假设是：输入变量 X 与输出变量 Y 之间存在一个线性关系，即：

$$Y = \beta_0 + \beta_1X$$

其中，$\beta_0$和$\beta_1$是模型的参数，分别表示截距和斜率。

逻辑回归模型的损失函数为：

$$L(\beta_0, \beta_1) = -\frac{1}{n}\sum_{i=1}^n[y_i\log(\hat{y_i}) + (1-y_i)\log(1-\hat{y_i})]$$


In [ ]:
class LogisticRegression:
    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    # 初始化参数
    def init_Params(self, n_features):
        self.weights = np.random.rand(n_features) * 0.01
        self.bias = 0.1

    def sigmoid(self, z):
        # z = wx + b
        return 1 / (1 + np.exp(-z))

    def compute_loss(self, y, y_pred):
        # 计算损失函数, 交叉熵损失函数
        # 参数是真实值和预测值
        epsilon = 1e-15
        # 防止log(0)
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
        return loss

    def fit(self, X, y):
        '''
        
        :param X: (n_samples,n_features) 输入数据 ， 
        :param y: (n_samples,1) 预测值 
        :return: self 训练好的模型 
        '''
        n_samples,n_features,  = X.shape
        self.init_Params(n_features)
        # 训练步骤：计算预测值，计算损失函数，更新参数
        for i in range(self.n_iters):
            # 计算预测值
            z = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(z)
            # 计算损失函数
            loss = self.compute_loss(y, y_pred)
            # 梯度计算公式:
            # ∂L/∂w = (1/n) * X^T · (y_pred - y)
            # ∂L/∂b = (1/n) * Σ (y_pred - y)
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)
            # 更新参数
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            # 打印损失函数
            if i % 100 == 0:
                print(f"Iteration: {i}, Loss: {loss}")

        return self

    # 预测属于某一类的概率，也就是计算预测值，按值分类
    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        y_pred = self.sigmoid(z)
        return y_pred

    # 预测属于某一类的标签
    def predict(self, X, threshold=0.5):
        '''
        假设y_pred_proba=[0.2, 0.6, 0.7]，阈值 threshold=0.5：
        条件判断结果：[False, True, True]
        最终输出：y_pred = [0, 1, 1]
        '''
        y_pred_proba = self.predict_proba(X)
        y_pred = np.where(y_pred_proba > threshold, 1, 0)
        return y_pred

    # 计算准确率
    def score(self, X, y):
        y_pred = self.predict(X)
        accuracy = np.mean(y_pred == y)
        # y_pred是[1,0,1]，而y是[1,1,0]，那么y_pred == y就会得到[True, False, False], 此时平均值就是准确率
        return accuracy




1. 生成模拟数据集...
数据集形状: X=(1000, 4), y=(1000,)
2. 数据预处理...
数据集形状: X_scaled=(1000, 4)
3. 划分训练集和测试集...
训练集形状: X_train=(800, 4), y_train=(800,)
测试集形状: X_test=(200, 4), y_test=(200,)
4. 训练模型...
Iteration: 0, Loss: 0.6918796784739653
Iteration: 100, Loss: 0.5685435661183956
Iteration: 200, Loss: 0.49196013226617497
Iteration: 300, Loss: 0.4415365409838803
Iteration: 400, Loss: 0.40637740918857546
Iteration: 500, Loss: 0.38066516243633314
Iteration: 600, Loss: 0.3611301057201564
Iteration: 700, Loss: 0.3458277997258016
Iteration: 800, Loss: 0.3335410007778967
Iteration: 900, Loss: 0.3234731744196385
模型训练完成...
5. 评估模型性能...
训练集准确率: 0.90
测试集准确率: 0.88
6. 查看模型参数...
模型权重: [ 1.7668306  -0.05293393 -0.03600072 -0.07784187]
模型偏置: 0.031942691037757515
7. 预测...
样本标签: [0 1 0 1 0 1 0 1 1 1]
样本预测标签: [0 0 0 1 0 1 0 0 1 0]
样本预测概率: [0.07922162 0.49737992 0.10380977 0.83374862 0.11044406 0.50953297
 0.10266202 0.45545516 0.95365132 0.39753025]


## <a id='toc1_3_'></a>[测试逻辑回归模型](#toc0_)

我们可以用 make_classification()函数生成一些随机数据，然后用逻辑回归模型进行训练和预测。

make_classification()能生成一些带有噪声的分类数据，包括有标签的数据和无标签的数据。

StandardScaler()可以对数据进行标准化处理，使得每个特征的方差为 1，均值为 0。

scaler.fit_transform(X)可以对数据进行标准化处理, 并返回标准化后的数据。

标准化公式如下，其中$\mu$是均值，$\sigma$是方差：

$$x_i' = \frac{x_i - \mu}{\sigma}$$

train_test_split()可以将数据集划分为训练集和测试集,接受的参数有：

- X: 输入数据
- y: 输出数据
- test_size: 测试集占比
- random_state: 随机种子


In [ ]:
def test_LogisticRegression():
    # 生成数据:
    print("1. 生成模拟数据集...")

    X, y = make_classification(
        n_samples=1000,  # 总样本数
        n_features=4,  # 特征数量
        n_informative=2,  # 有信息的特征数,其他两个没啥用
        n_redundant=0,  # 冗余特征数
        n_clusters_per_class=1,
        random_state=42  # 随机种子
    )
    print(f"数据集形状: X={X.shape}, y={y.shape}")

    # 数据预处理
    print("2. 数据预处理...")
    scaler = StandardScaler() # 得到标准化处理器
    X_scaled = scaler.fit_transform(X) # 标准化处理
    # 先计算均值与方差，再拿计算结果来标准化数据
    print(f"数据集形状: X_scaled={X_scaled.shape}")
    
    #划分 train/test 数据集
    print("3. 划分训练集和测试集...")
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    print(f"训练集形状: X_train={X_train.shape}, y_train={y_train.shape}")
    print(f"测试集形状: X_test={X_test.shape}, y_test={y_test.shape}")
     
     # 训练模型
    print("4. 训练模型...")
    model = LogisticRegression()
    model.fit(X_train, y_train)
    print("模型训练完成...")
    
    # 评估性能
    print("5. 评估模型性能...")
    train_accuracy = model.score(X_train, y_train)
    test_accuracy = model.score(X_test, y_test)
    print(f"训练集准确率: {train_accuracy:.2f}")
    print(f"测试集准确率: {test_accuracy:.2f}")
    
    # 查看模型参数
    print("6. 查看模型参数...")
    print(f"模型权重: {model.weights}")
    print(f"模型偏置: {model.bias}")
    
    # 预测
    print("7. 预测...")
    sample_indices = range(10)
    X_samples = X_test[sample_indices]
    y_samples = y_test[sample_indices]
    y_pred = model.predict(X_samples)
    y_pred_proba = model.predict_proba(X_samples)
    print(f"样本标签: {y_samples}")
    print(f"样本预测标签: {y_pred}")
    print(f"样本预测概率: {y_pred_proba}")
    

test_LogisticRegression()



1. 生成模拟数据集...
数据集形状: X=(1000, 4), y=(1000,)
2. 数据预处理...
数据集形状: X_scaled=(1000, 4)
3. 划分训练集和测试集...
训练集形状: X_train=(800, 4), y_train=(800,)
测试集形状: X_test=(200, 4), y_test=(200,)
4. 训练模型...
Iteration: 0, Loss: 0.6930739633201418
Iteration: 100, Loss: 0.5692697409022943
Iteration: 200, Loss: 0.492426018604295
Iteration: 300, Loss: 0.4418542957658561
Iteration: 400, Loss: 0.40660582767616704
Iteration: 500, Loss: 0.38083646137554794
Iteration: 600, Loss: 0.36126298801930923
Iteration: 700, Loss: 0.34593372850799914
Iteration: 800, Loss: 0.3336273415368959
Iteration: 900, Loss: 0.32354485448665643
模型训练完成...
5. 评估模型性能...
训练集准确率: 0.90
测试集准确率: 0.88
6. 查看模型参数...
模型权重: [ 1.76616264 -0.05231203 -0.03647102 -0.07772502]
模型偏置: 0.03183626294127122
7. 预测...
样本标签: [0 1 0 1 0 1 0 1 1 1]
样本预测标签: [0 0 0 1 0 1 0 0 1 0]
样本预测概率: [0.07923943 0.49730887 0.10394314 0.83364642 0.11040969 0.50919867
 0.10282298 0.45559895 0.95351549 0.39770715]
